In [1]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pickle

import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, MaxPool2D, Flatten

Load Data

In [2]:
def load_dataset(dataset_type="Type_01_(Raw_Gesture)"):
    """Load sign images and one-hot labels from the class-folder dataset.

    Images are resized to ``28x28`` and returned in (width, height, channels) format.
    The labels use the class order ``A-Z, DEL, SPACE``.
    """
    dataset_root =  Path("../../dataset/Root") / dataset_type
    class_names = [chr(label) for label in range(ord("A"), ord("Z") + 1)] + ["DEL", "SPACE"]
    images = []
    labels = []

    for class_index, class_name in enumerate(class_names):
        class_directory = dataset_root / class_name
        for image_path in sorted(class_directory.iterdir()):
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                continue

            image = cv2.resize(image, (28, 28)).astype(np.float32) / 255.0
            images.append(image)

            label = np.zeros(29, dtype=np.float32)
            label[class_index] = 1.0
            labels.append(label)

    if not images:
        raise FileNotFoundError(f"No readable images found in {dataset_root}")

    return np.array(images), np.array(labels)

x, y = load_dataset()

In [3]:
print(f"x shape: {x.shape}, y shape: {y.shape}")

x shape: (42000, 28, 28, 3), y shape: (42000, 29)


Data Split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)
print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Training set type: {type(X_train)}, Test set type: {type(X_test)}, Validation set type: {type(X_val)}")

Training set shape: (29400, 28, 28, 3), Test set shape: (6300, 28, 28, 3)
Validation set shape: (6300, 28, 28, 3)
Training set type: <class 'numpy.ndarray'>, Test set type: <class 'numpy.ndarray'>, Validation set type: <class 'numpy.ndarray'>


Setting up the CNN

In [5]:
model = Sequential()
model.add(Conv2D(32, (3, 3), padding = 'same', activation="relu"))
model.add(MaxPool2D((2, 2)))
model.add(Flatten())

model.add(Dense(29, activation="softmax"))

In [10]:
model.compile(optimizer = "adam",
              loss = "categorical_crossentropy",
              metrics = ["accuracy"])

In [11]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience = 5,
    restore_best_weights = True
)

In [12]:
model.fit(X_train, y_train,
epochs = 12,
batch_size = 16,
callbacks = [early_stop],
validation_data = (X_val, y_val))

Epoch 1/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9901 - loss: 0.0370 - val_accuracy: 0.9616 - val_loss: 0.1464
Epoch 2/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9933 - loss: 0.0291 - val_accuracy: 0.9713 - val_loss: 0.1061
Epoch 3/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9915 - loss: 0.0317 - val_accuracy: 0.9838 - val_loss: 0.0669
Epoch 4/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9939 - loss: 0.0234 - val_accuracy: 0.9835 - val_loss: 0.0650
Epoch 5/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9953 - loss: 0.0205 - val_accuracy: 0.9792 - val_loss: 0.0800
Epoch 6/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9944 - loss: 0.0216 - val_accuracy: 0.9840 - val_loss: 0.0653
Epoch 7/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9959 - loss: 0.0157 - val_accuracy: 0.9689 - val_loss: 0.1053
Epoch 8/12
1838/1838 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9973 - loss: 0.01

In [13]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9876 - loss: 0.0560
Test loss: 0.055982865393161774
Test accuracy: 0.9876190423965454


In [ ]:
y_pred = model.predict(X_test)
#changing one-hot encoded labels into classes
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
)
#accuracy
accuracy = np.mean(y_pred_classes == y_test_classes)
print("Accuracy:", accuracy)

# Confusion matrix
cm = confusion_matrix(y_test_classes, y_pred_classes)
print("\nConfusion Matrix:")
print(cm)

# Precision, Recall, F1
print("\nClassification Report:")
print(classification_report(y_test_classes, y_pred_classes))

Accuracy: 0.9839285714285714

Confusion Matrix:
[[299   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    1   0   0   0   0   0   0   0   0   0]
 [  0 297   0   1   0   0   0   0   0   0   1   0   0   0   0   0   0   0
    0   0   1   0   0   0   0   0   0   0]
 [  0   0 297   0   1   0   0   0   0   0   0   1   0   0   0   0   0   0
    1   0   0   0   0   0   0   0   0   0]
 [  0   0   0 298   0   0   0   0   2   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0 296   0   1   0   0   0   0   0   0   0   0   0   0   0
    2   0   0   0   0   0   1   0   0   0]
 [  0   1   0   1   0 294   0   0   0   0   0   0   0   0   0   0   0   1
    0   2   0   0   1   0   0   0   0   0]
 [  1   0   0   0   0   0 298   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   1   0   0]
 [  0   0   0   0   0   0   1 296   0   0   0   0   0   1   0   0   0   0
    0   0   0   0   0   0   0   2   0   0]
 [  0   1   0   

In [ ]:
with open("models/comparator_model", 'wb') as files:
    pickle.dump(model, files)